## 1. 爬楼梯

- **难度**：简单  
- **标签**：动态规划

**题目**：假设你正在爬楼梯。需要 n 阶你才能到达楼顶。每次你可以爬 1 或 2 个台阶。你有多少种不同的方法可以爬到楼顶呢？

**示例**：
```
输入：n = 2
输出：2
解释：1 阶 + 1 阶；2 阶

输入：n = 3
输出：3
解释：1+1+1、1+2、2+1
```

**思路**：dp[i] 表示爬到第 i 阶的方法数。最后一步只有两种情况：从 i-1 爬 1 阶、从 i-2 爬 2 阶，故 dp[i] = dp[i-1] + dp[i-2]（斐波那契式）。优化：数组可压缩为 prev1/prev2 两个滚动变量。时间 O(n)，空间 O(1)。

**亮点**：从「最后一步怎么走」出发做状态转移，是线性 DP 的入门范式；状态压缩（滚动变量）在只依赖前两个状态时直接可用。


In [ ]:
class Solution:
    def climbStairs(self, n: int) -> int:
        # n = 1 时，只有一种爬法
        if n == 1:
            return 1

        # prev2 表示前前一个状态（对应 dp[0]）
        prev2 = 1
        # prev1 表示前一个状态（对应 dp[1]）
        prev1 = 1

        # 从第 2 阶开始计算
        for i in range(2, n + 1):
            # 当前状态 = 前一个状态 + 前前一个状态
            current = prev1 + prev2

            # 状态向前滚动：原来的 prev1 变成 prev2
            prev2 = prev1
            prev1 = current

        # 循环结束后，prev1 就是 dp[n]
        return prev1


class Solution2:
    def climbStairs(self, n: int) -> int:
        # dp[i] 表示：爬到第 i 阶楼梯一共有多少种不同的方法
        dp = [0] * (n + 1)
        dp[0] = 1   # 爬到第 0 阶，可理解为"什么都不做"
        dp[1] = 1   # 爬到第 1 阶只有一种方法

        for i in range(2, n + 1):
            # 最后一步：从 i-1 爬 1 阶 或 从 i-2 爬 2 阶
            dp[i] = dp[i - 1] + dp[i - 2]

        return dp[n]


# 测试
sol = Solution()
print(sol.climbStairs(2))    # 2
print(sol.climbStairs(3))    # 3
print(sol.climbStairs(45))   # 1836311903

sol2 = Solution2()
print(sol2.climbStairs(2))   # 2
print(sol2.climbStairs(10))  # 89


## 2. 最大子数组和

- **难度**：中等  
- **标签**：动态规划

**题目**：给你一个整数数组 nums，请你找出一个具有最大和的连续子数组（子数组最少包含一个元素），返回其最大和。子数组是数组中的一个连续部分。

**示例**：
```
输入：nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
输出：6
解释：连续子数组 [4, -1, 2, 1] 的和最大，为 6。
```

**思路**：dp[i] 表示以 i 为右端点的最大子数组和。每个位置二选一：接上之前的最优段（dp[i-1] + nums[i]），或从 nums[i] 重新开始（nums[i]），取较大值。答案遍历时取所有 dp[i] 的最大值。时间 O(n)，空间 O(n)（可滚动到 O(1)）。

**亮点**：状态定义「以 i 结尾」是子数组类 DP 的标准姿势——把「枚举所有子数组」O(n²) 降为「每个位置只做一次二选一」O(n)，即 Kadane 算法。


In [ ]:
from typing import List

class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
        # 动态规划
        # 定义 dp[i]：以 i 为右端点的数组的最大子数组和
        n = len(nums)

        dp = [0] * n
        dp[0] = nums[0]

        ans = nums[0]  # 记录所有 dp[i] 中的最大值

        for i in range(1, n):
            # 要么接上 dp[i-1] 的段，要么从 nums[i] 重新开始
            dp[i] = max(dp[i - 1] + nums[i], nums[i])

            ans = max(ans, dp[i])

        return ans


# 测试
sol = Solution()
print(sol.maxSubArray([-2, 1, -3, 4, -1, 2, 1, -5, 4]))  # 6
print(sol.maxSubArray([1]))                              # 1
print(sol.maxSubArray([5, 4, -1, 7, 8]))                 # 23


## 3. 打家劫舍

- **难度**：中等  
- **标签**：动态规划

**题目**：你是一个专业的小偷，计划偷窃沿街的房屋。每间房内都藏有一定的现金，影响你偷窃的唯一制约因素就是相邻的房屋装有相互连通的防盗系统，如果两间相邻的房屋在同一晚上被小偷闯入，系统会自动报警。给定一个代表每个房屋存放金额的非负整数数组，计算你**不触动警报装置**的情况下，一夜之内能够偷窃到的最高金额。

**示例**：
```
输入：nums = [1, 2, 3, 1]
输出：4
解释：偷 1 号（1）和 3 号（3），共 4。

输入：nums = [2, 7, 9, 3, 1]
输出：12
解释：偷 1 号（2）、3 号（9）、5 号（1），共 12。
```

**思路**：dp[i] 表示考虑前 i 间房屋能偷到的最高金额。最后一间房屋：不偷 → dp[i-1]；偷 → 第 i-1 间不能偷，得 dp[i-2] + nums[i]。取较大值。时间 O(n)，空间 O(n)（可滚动到 O(1)）。

**亮点**：经典的「偷 or 不偷」0-1 决策模型——状态转移只依赖前两个状态 dp[i-1]、dp[i-2]，与爬楼梯结构同源，但决策从「求和」变成「取 max」。


In [ ]:
from typing import List

class Solution:
    def rob(self, nums: List[int]) -> int:
        n = len(nums)

        # 如果没有房屋，偷到的金额为 0
        if n == 0:
            return 0

        # 如果只有一间房屋，只能偷这一间
        if n == 1:
            return nums[0]

        # dp[i] 表示考虑前 i 间房屋时，能够偷到的最高金额
        dp = [0] * n
        dp[0] = nums[0]
        dp[1] = max(nums[0], nums[1])

        # 枚举每一间房屋，考虑最后一间房屋偷还是不偷
        for i in range(2, n):
            # 不偷第 i 间：答案是 dp[i - 1]
            # 偷第 i 间：第 i - 1 间不能偷，答案是 dp[i - 2] + nums[i]
            dp[i] = max(dp[i - 1], dp[i - 2] + nums[i])

        return dp[n - 1]


# 测试
sol = Solution()
print(sol.rob([1, 2, 3, 1]))        # 4
print(sol.rob([2, 7, 9, 3, 1]))     # 12
print(sol.rob([0]))                 # 0


## 4. 最长递增子序列

- **难度**：中等  
- **标签**：动态规划

**题目**：给你一个整数数组 nums，找到其中**最长严格递增子序列**的长度。子序列是由数组派生而来的序列，删除（或不删除）数组中的元素而不改变其余元素的顺序。

**示例**：
```
输入：nums = [10, 9, 2, 5, 3, 7, 101, 18]
输出：4
解释：最长递增子序列是 [2, 3, 7, 101]，长度为 4。

输入：nums = [0, 1, 0, 3, 2, 3]
输出：4
输入：nums = [7, 7, 7, 7, 7, 7, 7]
输出：1
```

**思路**：dp[i] 表示以 nums[i] 作为结尾的最长递增子序列长度（初始为 1）。对每个 i，枚举它前面的所有 j：若 nums[j] < nums[i]，则 nums[i] 可以接在 nums[j] 后面，dp[i] = max(dp[i], dp[j] + 1)。答案取所有 dp[i] 的最大值。时间 O(n²)，空间 O(n)。

**亮点**：「以 i 结尾」的状态定义使转移只需考虑「前面哪个元素可以当倒数第二个」；注意 dp[i] 是「必须包含 nums[i]」的局部值，全局答案仍是 max——与最大子数组和的答案维护方式一致。可优化：第二层用二分将复杂度降为 O(n log n)。


In [ ]:
from typing import List

class Solution:
    def lengthOfLIS(self, nums: List[int]) -> int:
        n = len(nums)

        # dp[i] 表示以 nums[i] 作为结尾的最长递增子序列长度
        dp = [1] * n

        # ans 记录所有 dp[i] 中的最大值
        ans = 1

        # 从左到右枚举每个位置作为子序列结尾
        for i in range(n):
            # 枚举 i 前面的所有位置
            for j in range(i):
                # 只有 nums[j] < nums[i]，nums[i] 才能接在 nums[j] 后面
                if nums[j] < nums[i]:
                    dp[i] = max(dp[i], dp[j] + 1)

            # 更新最长递增子序列长度
            ans = max(ans, dp[i])

        return ans


# 测试
sol = Solution()
print(sol.lengthOfLIS([10, 9, 2, 5, 3, 7, 101, 18]))  # 4
print(sol.lengthOfLIS([0, 1, 0, 3, 2, 3]))            # 4
print(sol.lengthOfLIS([7, 7, 7, 7, 7, 7, 7]))         # 1
